# SI26 — Week 7: Language ID Model (Code-Switching) — Train + Deploy

### Code Saviours Summer Internship 2026 — Project 2, Phase 3
**Author:** Qandeel Asim

**Goal:** Train a token-classification model that labels each word in Roman Urdu text as `URD`, `ENG`, or `MIX`, then publish it to Hugging Face Hub.

**Uses your actual Week 6 dataset** (`code-switching-codesaviours-si26-qandeel` — 3182 word-rows / 200 sentences, from the Sharf 2017 Roman Urdu Data Set).

**Before running:**
1. `Runtime > Change runtime type > GPU`
2. Run cells top to bottom, in order — each cell depends on the one before it
3. Have your Hugging Face **write-access token** ready (huggingface.co/settings/tokens)

**Submit by Friday (paste in Classroom):**
- Hugging Face Model Hub link for your published model
- This Week 7 notebook link on GitHub
- Evaluation scores — F1 for URD, ENG, MIX labels


## Step 0 — Setup

In [1]:
# Pinning transformers avoids the ImportError from Week 4 (fixed in v8) —
# keep this pin unless you have a specific reason to bump it.
!pip install -q transformers==4.46.3 torch datasets seqeval scikit-learn huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 90.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — go to Runtime > Change runtime type > GPU')


GPU available: True
Device name: Tesla T4


## Step 1 — Load Your Week 6 Dataset

This tries, in order:
1. `dataset.csv` already sitting in this Colab session (only works if you're continuing straight from Week 6 without restarting)
2. Your GitHub repo (`code-switching-codesaviours-si26-qandeel`) raw CSV
3. Your Hugging Face dataset (`code-switching-codesaviours-si26-qandeel`)
4. Manual upload, as a last resort

No manual path-editing needed unless all four fail.

In [3]:
import pandas as pd
import os

GITHUB_USERNAME = 'qandeelasim13'        # <-- edit if different
HF_USERNAME = 'qandeelasim13'            # <-- edit to your actual HF username
DATASET_REPO_NAME = 'code-switching-codesaviours-si26-qandeel'

df = None

# keep_default_na=False / na_filter=False stops pandas from silently turning words
# like 'null', 'NA', 'None' (real tokens in informal Roman Urdu text) into NaN floats —
# that mismatch is what causes the tokenizer TypeError further down if left unguarded.
CSV_READ_KWARGS = dict(keep_default_na=False, na_filter=False, dtype=str)

# Attempt 1 — local file left over from Week 6 (same session)
if os.path.exists('dataset.csv'):
    try:
        df = pd.read_csv('dataset.csv', **CSV_READ_KWARGS)
        print('Loaded dataset.csv from local Colab session.')
    except Exception as e:
        print(f'Local dataset.csv found but failed to read: {e}')

# Attempt 2 — GitHub raw CSV
if df is None:
    github_url = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/{DATASET_REPO_NAME}/main/dataset.csv'
    try:
        df = pd.read_csv(github_url, **CSV_READ_KWARGS)
        print(f'Loaded dataset from GitHub: {github_url}')
    except Exception as e:
        print(f'GitHub load failed ({e}). Trying Hugging Face...')

# Attempt 3 — Hugging Face Hub dataset
if df is None:
    try:
        from datasets import load_dataset
        hf_ds = load_dataset(f'{HF_USERNAME}/{DATASET_REPO_NAME}')
        split_name = list(hf_ds.keys())[0]
        df = hf_ds[split_name].to_pandas().astype(str)
        print(f'Loaded dataset from Hugging Face Hub: {HF_USERNAME}/{DATASET_REPO_NAME}')
    except Exception as e:
        print(f'Hugging Face load failed ({e}).')

# Attempt 4 — manual upload fallback
if df is None:
    print('\nAll automatic sources failed. Please upload dataset.csv manually.')
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0], **CSV_READ_KWARGS)

# Belt-and-suspenders: whatever the source, force word/sentence/label to plain str
# and drop any genuinely empty word rows (guards against the tokenizer TypeError
# that happens if a non-string value sneaks into the 'words' lists later on).
for col in ['sentence', 'word', 'label']:
    df[col] = df[col].astype(str)
df = df[df['word'].str.len() > 0].reset_index(drop=True)

print(f'\nRows loaded: {len(df)}')
df.head()


GitHub load failed (HTTP Error 404: Not Found). Trying Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4816 [00:00<?, ? examples/s]

Loaded dataset from Hugging Face Hub: qandeelasim13/code-switching-codesaviours-si26-qandeel

Rows loaded: 4816


,sentence,word,label
0,Taham Baba e Urdu ki marqa nigari is saqam se ...,Taham,URD
1,Taham Baba e Urdu ki marqa nigari is saqam se ...,Baba,URD
2,Taham Baba e Urdu ki marqa nigari is saqam se ...,e,URD
3,Taham Baba e Urdu ki marqa nigari is saqam se ...,Urdu,URD
4,Taham Baba e Urdu ki marqa nigari is saqam se ...,ki,URD


In [17]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1WCV2hMbQN2PHcYH9hJfOWUkH_cwnDNkye74e_bSER8o/edit#gid=0


In [4]:
# Sanity checks — catches column/label problems before they become confusing training errors
required_cols = {'sentence', 'word', 'label'}
missing = required_cols - set(df.columns)
assert not missing, f'dataset.csv is missing required columns: {missing}'

label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

bad_labels = set(df['label'].unique()) - set(label2id.keys())
assert not bad_labels, f'Found labels not in {{URD, ENG, MIX}}: {bad_labels}'

print('Label distribution:')
print(df['label'].value_counts())
print(f"\nUnique sentences: {df['sentence'].nunique()}")
print(f"Total word rows: {len(df)}")

mix_count = (df['label'] == 'MIX').sum()
if mix_count < 30:
    print(f"\n⚠️  Only {mix_count} MIX-labelled words in the whole dataset.")
    print('   After an 80/20 split, the test set may end up with very few (or zero) MIX')
    print('   examples — MIX F1 could come out as 0.0 or NaN purely from sample size, not')
    print('   a real model failure. If that happens, mention it in your submission comment')
    print("   rather than assuming the model is broken. (Optional fix: relabel a few more")
    print("   loanwords like 'internet', 'message', 'order' as MIX in dataset.csv before")
    print('   re-running, to give the model more to learn from.)')


Label distribution:
label
URD    4367
ENG     271
MIX     178
Name: count, dtype: int64

Unique sentences: 300
Total word rows: 4816


## Step 2 — Prepare Sentences for Token Classification

In [5]:
# Cell 1 — Group into sentences, then split train/test stratified on MIX presence.
# Why: MIX is so rare that a plain random split can dump nearly all MIX examples into
# one side (train or test), leaving the other with 0-1 — which is exactly what caused
# MIX F1 to be stuck at 0.000 despite good training. Splitting MIX-containing and
# non-MIX sentences separately (then combining) guarantees MIX shows up on both sides.
from sklearn.model_selection import train_test_split
import random

sentences = []
for _, g in df.groupby('sentence', sort=False):
    sentences.append({'words': g['word'].tolist(), 'labels': g['label'].tolist()})

print(f'Total sentences: {len(sentences)}')

mix_sentences = [s for s in sentences if 'MIX' in s['labels']]
other_sentences = [s for s in sentences if 'MIX' not in s['labels']]
print(f'Sentences containing MIX: {len(mix_sentences)}')

if len(mix_sentences) >= 5:
    mix_train, mix_test = train_test_split(mix_sentences, test_size=0.2, random_state=42)
else:
    # Too few to split meaningfully — put them all in train so the model at least
    # sees them during training; test set MIX support will still be near 0.
    mix_train, mix_test = mix_sentences, []
    print('Fewer than 5 MIX sentences — putting all of them in train. Consider running')
    print('SI26_Week6_MIXBoost_Qandeel.ipynb to get a dataset with more MIX coverage.')

other_train, other_test = train_test_split(other_sentences, test_size=0.2, random_state=42)

train_data = mix_train + other_train
test_data = mix_test + other_test
random.Random(42).shuffle(train_data)
random.Random(42).shuffle(test_data)

print(f'Training sentences: {len(train_data)} (MIX: {len(mix_train)})')
print(f'Testing sentences: {len(test_data)} (MIX: {len(mix_test)})')


Total sentences: 300
Sentences containing MIX: 149
Training sentences: 239 (MIX: 119)
Testing sentences: 61 (MIX: 30)


## Step 3 — Fine-tune XLM-RoBERTa for Token Classification

In [6]:
# Cell 2a — Model, tokenizer, tokenize+align labels
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset

model_name = 'xlm-roberta-base'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def tokenize_and_align_labels(examples):
    # Force every word to a plain python str, in a plain python list of lists.
    # This is the actual fix for: TypeError: PreTokenizedEncodeInput must be
    # Union[PreTokenizedInputSequence, Tuple[...]] — that error fires when any
    # word slips through as a non-str (NaN/float) or as a numpy/Arrow-backed
    # type instead of a native str, which the Rust tokenizer rejects outright.
    words_batch = [[str(w) for w in ws] for ws in examples['words']]
    tokenized = tokenizer(words_batch, truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data]
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/239 [00:00<?, ? examples/s]

Map:   0%|          | 0/61 [00:00<?, ? examples/s]

In [7]:
# Cell 2b — Metrics: per-label F1 (URD/ENG/MIX) required for your submission comment
#
# IMPORTANT: seqeval is built for NER-style B-/I- tagged spans. Our labels
# (URD/ENG/MIX) are plain per-word tags, not BIO spans — feeding them to seqeval
# silently produces 0.0 for every per-label F1 (you'll see 'X seems not to be
# NE tag' warnings if you ever go back to that approach). We use sklearn's
# token-level precision_recall_fscore_support instead, which is the correct
# tool for a flat multi-class per-token classification problem like this one.
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix

LABEL_NAMES = ['URD', 'ENG', 'MIX']

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions, true_labels = [], []
    for pred, label in zip(predictions, labels):
        for p, l in zip(pred, label):
            if l != -100:
                true_predictions.append(id2label[p])
                true_labels.append(id2label[l])

    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, true_predictions, labels=LABEL_NAMES, average=None, zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average='weighted', zero_division=0
    )

    metrics = {'overall_f1': weighted_f1}
    for i, lbl in enumerate(LABEL_NAMES):
        metrics[f'f1_{lbl}'] = f1[i]
        metrics[f'precision_{lbl}'] = precision[i]
        metrics[f'recall_{lbl}'] = recall[i]
        metrics[f'support_{lbl}'] = int(support[i])
    return metrics


### Handling the class imbalance (URD 2936 / ENG 234 / MIX 12)

Left alone, a model trained on this split mostly just learns "predict URD" and still
scores well on paper because URD is 92% of the data. To get a model that's actually
useful for ENG and MIX, not just URD, we weight the loss so mistakes on the rare
classes count more during training.

In [8]:
# Cell 2c — Class-weighted loss (so the model doesn't just learn to predict URD everywhere)
import torch
import torch.nn as nn

label_counts = df['label'].value_counts()
total_words = label_counts.sum()
num_classes = len(LABEL_NAMES)

raw_weights = {
    lbl: total_words / (num_classes * label_counts.get(lbl, 1))
    for lbl in LABEL_NAMES
}
# Cap the max weight — MIX has so few examples (12) that its raw inverse-frequency
# weight would be huge and destabilize training. Capping keeps it influential
# without letting it dominate the loss.
MAX_WEIGHT = 15.0
class_weights = torch.tensor(
    [min(raw_weights[lbl], MAX_WEIGHT) for lbl in LABEL_NAMES],
    dtype=torch.float
)

print('Class weights (URD, ENG, MIX):', class_weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device), ignore_index=-100)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


Class weights (URD, ENG, MIX): [0.3676055371761322, 5.923739433288574, 9.018726348876953]


In [9]:
# Cell 2d — Training args + WeightedTrainer
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=8,                 # bumped from 5 — small dataset, benefits from more passes
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',        # transformers 4.46.3 renamed evaluation_strategy -> eval_strategy
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='overall_f1',
    greater_is_better=True,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()
print('Training complete!')


/tmp/ipykernel_519/4040939328.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


Starting training...


Epoch,Training Loss,Validation Loss,Overall F1,F1 Urd,Precision Urd,Recall Urd,Support Urd,F1 Eng,Precision Eng,Recall Eng,Support Eng,F1 Mix,Precision Mix,Recall Mix,Support Mix
1,1.097100,1.000911,0.518226,0.562343,1.000000,0.391153,859,0.161919,0.088091,1.000000,54,0.000000,0.000000,0.000000,36
2,0.717500,0.548234,0.833145,0.879064,0.995582,0.786962,859,0.426829,0.318182,0.648148,54,0.346939,0.212500,0.944444,36
3,0.450000,0.424965,0.883630,0.923845,0.995962,0.861467,859,0.497041,0.365217,0.777778,54,0.503937,0.351648,0.888889,36
4,0.309500,0.418197,0.892728,0.931931,0.994716,0.876601,859,0.525641,0.401961,0.759259,54,0.507937,0.355556,0.888889,36
5,0.276100,0.386954,0.907805,0.941825,0.993540,0.895227,859,0.604027,0.473684,0.833333,54,0.551724,0.400000,0.888889,36
6,0.206600,0.375948,0.919599,0.950939,0.991162,0.913853,859,0.661765,0.548780,0.833333,54,0.558559,0.413333,0.861111,36
7,0.195500,0.360919,0.923096,0.952727,0.993679,0.915017,859,0.680851,0.551724,0.888889,54,0.579439,0.436620,0.861111,36
8,0.159400,0.355956,0.921570,0.952208,0.991184,0.916182,859,0.643357,0.516854,0.851852,54,0.607843,0.469697,0.861111,36


Training complete!


In [10]:
# Cell 2e — Final evaluation: F1 scores + full classification report + confusion matrix
eval_results = trainer.evaluate()
print('Final evaluation metrics:')
for k, v in eval_results.items():
    print(f'  {k}: {v}')

print('\n--- Copy these into your submission comment ---')
print(f"F1 URD: {eval_results.get('eval_f1_URD', 'N/A')}")
print(f"F1 ENG: {eval_results.get('eval_f1_ENG', 'N/A')}")
print(f"F1 MIX: {eval_results.get('eval_f1_MIX', 'N/A')}")
print(f"Overall (weighted) F1: {eval_results.get('eval_overall_f1', 'N/A')}")

# Full breakdown — useful for your submission write-up and for Demo Day
predictions, labels, _ = trainer.predict(test_ds)
predictions = np.argmax(predictions, axis=2)

flat_preds, flat_labels = [], []
for pred, label in zip(predictions, labels):
    for p, l in zip(pred, label):
        if l != -100:
            flat_preds.append(id2label[p])
            flat_labels.append(id2label[l])

print('\n--- Full classification report ---')
print(classification_report(flat_labels, flat_preds, labels=LABEL_NAMES, zero_division=0))

print('--- Confusion matrix (rows = true, cols = predicted) ---')
cm = confusion_matrix(flat_labels, flat_preds, labels=LABEL_NAMES)
print('        ' + '  '.join(f'{l:>6}' for l in LABEL_NAMES))
for lbl, row in zip(LABEL_NAMES, cm):
    print(f'{lbl:>6}  ' + '  '.join(f'{v:>6}' for v in row))

mix_test_count = flat_labels.count('MIX')
if mix_test_count == 0:
    print(f"\nNote: the test split had 0 MIX-labelled tokens, so MIX F1 is undefined by")
    print('sample size, not a model failure — mention this in your submission comment.')


Final evaluation metrics:
  eval_loss: 0.36091870069503784
  eval_overall_f1: 0.923096414966963
  eval_f1_URD: 0.9527272727272728
  eval_precision_URD: 0.9936788874841972
  eval_recall_URD: 0.9150174621653085
  eval_support_URD: 859
  eval_f1_ENG: 0.6808510638297872
  eval_precision_ENG: 0.5517241379310345
  eval_recall_ENG: 0.8888888888888888
  eval_support_ENG: 54
  eval_f1_MIX: 0.5794392523364486
  eval_precision_MIX: 0.43661971830985913
  eval_recall_MIX: 0.8611111111111112
  eval_support_MIX: 36
  eval_runtime: 0.2591
  eval_samples_per_second: 235.437
  eval_steps_per_second: 30.877
  epoch: 8.0

--- Copy these into your submission comment ---
F1 URD: 0.9527272727272728
F1 ENG: 0.6808510638297872
F1 MIX: 0.5794392523364486
Overall (weighted) F1: 0.923096414966963

--- Full classification report ---
              precision    recall  f1-score   support

         URD       0.99      0.92      0.95       859
         ENG       0.55      0.89      0.68        54
         MIX       0.

## Step 3 — Try It Out (Demo Day)

In [11]:
# Quick inference helper — type any Roman Urdu / English sentence and see word-level predictions
def predict_labels(sentence: str):
    words = sentence.split()
    inputs = tokenizer(words, is_split_into_words=True, truncation=True, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    preds = torch.argmax(logits, dim=2)[0].tolist()
    word_ids = tokenizer(words, is_split_into_words=True, truncation=True).word_ids()

    result = []
    prev_word = None
    for word_id, pred in zip(word_ids, preds):
        if word_id is None or word_id == prev_word:
            continue
        result.append((words[word_id], id2label[pred]))
        prev_word = word_id
    return result

demo_sentences = [
    'Mujhe ye phone bohot pasand aya',
    'Please call me when you are free bhai',
    'Overall service was good lekin thora slow tha',
]

for s in demo_sentences:
    print(s)
    for word, label in predict_labels(s):
        print(f'  {word:15s} -> {label}')
    print()


Mujhe ye phone bohot pasand aya
  Mujhe           -> URD
  ye              -> URD
  phone           -> MIX
  bohot           -> URD
  pasand          -> URD
  aya             -> URD

Please call me when you are free bhai
  Please          -> ENG
  call            -> MIX
  me              -> ENG
  when            -> ENG
  you             -> ENG
  are             -> ENG
  free            -> MIX
  bhai            -> URD

Overall service was good lekin thora slow tha
  Overall         -> ENG
  service         -> MIX
  was             -> ENG
  good            -> ENG
  lekin           -> URD
  thora           -> URD
  slow            -> ENG
  tha             -> URD



## Step 4 — Save and Push to Hugging Face Hub

In [12]:
# Cell 3a — Login (paste your HuggingFace WRITE token when prompted)
from huggingface_hub import notebook_login
notebook_login()


In [14]:
# Cell 3b — Push model + tokenizer
# Using the same repo-name convention as your Week 6 dataset repo
repo_name = 'code-switching-codesaviours-si26-qandeel'

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f'Model published at: https://huggingface.co/{HF_USERNAME}/{repo_name}')


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...btblc52/model.safetensors:   0%|          | 13.6kB / 1.11GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpuyepda1r/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

  ...r/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model published at: https://huggingface.co/qandeelasim13/code-switching-codesaviours-si26-qandeel


In [15]:
# Cell 3c — Push a proper model card (README) so the Hub page isn't blank
from huggingface_hub import HfApi

model_card = f"""---
language: ur
tags:
- token-classification
- code-switching
- roman-urdu
license: cc-by-4.0
---

# Code-Switching Language ID — Roman Urdu / English

Token classification model that labels each word in Roman Urdu text as `URD` (Roman Urdu),
`ENG` (English), or `MIX` (nativized English loanword used as Urdu vocabulary).

Fine-tuned from [xlm-roberta-base](https://huggingface.co/xlm-roberta-base).

## Dataset
- Source: filtered from the [Roman Urdu Data Set](https://archive.ics.uci.edu/dataset/458/roman+urdu+data+set) (Sharf, 2017, UCI ML Repository, CC BY 4.0)
- 200 sentences, 3182 word-level labels
- Label distribution: URD {label_counts.get('URD', 0)}, ENG {label_counts.get('ENG', 0)}, MIX {label_counts.get('MIX', 0)}
- Dataset card: https://huggingface.co/datasets/{HF_USERNAME}/{repo_name}

## Evaluation (held-out 20% split)
| Label | F1 | Precision | Recall | Support |
|---|---|---|---|---|
| URD | {eval_results.get('eval_f1_URD', 0):.3f} | {eval_results.get('eval_precision_URD', 0):.3f} | {eval_results.get('eval_recall_URD', 0):.3f} | {eval_results.get('eval_support_URD', 0)} |
| ENG | {eval_results.get('eval_f1_ENG', 0):.3f} | {eval_results.get('eval_precision_ENG', 0):.3f} | {eval_results.get('eval_recall_ENG', 0):.3f} | {eval_results.get('eval_support_ENG', 0)} |
| MIX | {eval_results.get('eval_f1_MIX', 0):.3f} | {eval_results.get('eval_precision_MIX', 0):.3f} | {eval_results.get('eval_recall_MIX', 0):.3f} | {eval_results.get('eval_support_MIX', 0)} |

Overall weighted F1: {eval_results.get('eval_overall_f1', 0):.3f}

## Limitations
- MIX is severely underrepresented in training data (12 of 3182 word labels), so MIX
  predictions should be treated as low-confidence until the dataset is expanded.
- Trained on informal social-media Roman Urdu; may not generalize well to formal text.

## Training
XLM-RoBERTa-base fine-tuned for 8 epochs with a class-weighted loss (weights: URD
{class_weights[0]:.2f}, ENG {class_weights[1]:.2f}, MIX {class_weights[2]:.2f}) to
counter the class imbalance above.

Built for Code Saviours Summer Internship 2026 (SI-26), Week 7.
"""

with open('MODEL_CARD.md', 'w') as f:
    f.write(model_card)

api = HfApi()
api.upload_file(
    path_or_fileobj='MODEL_CARD.md',
    path_in_repo='README.md',
    repo_id=f'{HF_USERNAME}/{repo_name}',
    repo_type='model',
)
print('Model card pushed.')


Model card pushed.


In [16]:
# Optional but recommended — save eval scores locally too, so you have a record
# even before you paste them into Classroom
import json

with open('week7_eval_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print('Saved week7_eval_results.json')


Saved week7_eval_results.json


## Resources
- XLM-RoBERTa: https://huggingface.co/xlm-roberta-base
- Token classification tutorial: https://huggingface.co/docs/transformers/tasks/token_classification
- HuggingFace Hub push: https://huggingface.co/docs/hub/models-uploading

## Submission Checklist
- [ ] Hugging Face Model Hub link for your published model (now with a proper model card / README)
- [ ] Week 7 notebook pushed to GitHub, link pasted in Classroom
- [ ] F1 scores for URD, ENG, MIX (and overall weighted F1) pasted in your submission comment — copy from the "Final evaluation metrics" + classification report output, not the training-log numbers
- [ ] If MIX F1 is 0/undefined, note in your submission comment that the test split had ~0 MIX examples (only 12 exist in the whole dataset) — this is a data-size limitation, not a training bug
